New Notebook Created by Jupyter MCP Server

In [1]:
from pathlib import Path
import sys, os
ROOT = Path('/home/nahisaho/GitHub/jupytermind')
sys.path.insert(0, str(ROOT / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(ROOT / 'benchmark/projects')
from ai_data_scientist import project_manager
handle = project_manager.resolve_project('kaggle-v020-kaggle-exp-22-seaice')
project_manager.ensure_notebook(handle)
print('保存先:', handle.notebook_path)
print('kernel cwd:', Path.cwd())

保存先: /home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-22-seaice/notebooks/kaggle-v020-kaggle-exp-22-seaice.ipynb
kernel cwd: /home/nahisaho/GitHub/jupytermind


In [14]:
# Notebookの整理はenqueue_writeで直列化。分析コードの実行は引き続きMCPセル実行のみ。
final_source = r"""# 最終記録・図表監査・ライフサイクル（再実行時も根拠参照を更新）
with phase('改善候補の最小再現とNotebook文書化'):
    inference_probe = data_definition.build_manifest({}, {}, {'x':{'unit':data_definition.FieldValue('推定単位','inferred','試験')}})
    probe_inferred_not_reported = inference_probe.unresolved_fields() == []
    visual_probe = nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell('display(chart)',execution_count=1,
        outputs=[visualization.build_image_output((data_dir/'01-annual-trend.png').read_bytes())])])
    visual_probe.cells[0].outputs[0]['execution_count']=1
    probe_visual_metadata_not_reported = notebook_audit.audit_visual_outputs(visual_probe,(0,)) == ()
    print('改善候補の再現:', {'inferred_not_in_unresolved_fields':probe_inferred_not_reported,'missing_chart_metadata_not_flagged':probe_visual_metadata_not_reported})
    notebook_before=nbformat.read(handle.notebook_path,as_version=4)
    evidence_cell=next(c for c in notebook_before.cells if c.cell_type=='code' and c.source.startswith('# 根拠集約:'))
    evidence_execution_count=evidence_cell.execution_count
    evidence_text='\n'.join(str(o.get('data',{}).get('text/plain','')) for o in evidence_cell.outputs)
    result={'output':evidence_text}
    claim_definitions=[
        ('annual_slope','長期傾向：1979–2018年の完全年度38年では、月等重み年平均は10年あたり約0.546百万km²減少した。年HAC(3)95%区間は−0.601〜−0.491百万km²/10年。全年度が単調に減るという意味ではない。','descriptive-trend'),
        ('seasonal_peak_to_trough','季節変動：平均的季節成分は3月最大、9月最小で、最大–最小差は約9.200百万km²。これは全期間の平均形であり、各年の季節形が一定という主張ではない。','descriptive-seasonality'),
        ('september_slope','季節別の変化：9月は約−0.810、3月は約−0.416百万km²/10年。夏の最小期の減少の方が大きい。全12か月の点推定も負だった。12個の区間は同時信頼区間ではない。','descriptive-seasonal-trend'),
        ('amplitude_slope','季節振幅の変化：同じ年の3月–9月差は約+0.394百万km²/10年増加し、HAC(3)95%区間は+0.222〜+0.566。固定月効果による分解は平均的な基準表示に限定し、この変化を併記する。','descriptive-amplitude-change'),
        ('sensitivity_max_relative_deviation','結論安定性：24仕様で年平均の傾きは−0.638〜−0.546百万km²/10年、すべて減少方向。stable=True、基準からの最大相対差は0.1694、事前に指定した20%許容内。ただし全ての可能な仕様への頑健性を保証しない。','sensitivity'),
    ]
    extracted_claims=[(insight_engine.extract_cited_value(result, '"'+key+r'":\s*([-0-9.e]+)'),text,kind) for key,text,kind in claim_definitions]
    modules_text='''## 使用したJupytermindモジュール
直接import・呼び出したものだけを列挙する。間接利用は含めない。ローカル関数phase/show_chart/sensitivity_analysis/wild_block_intervalはNotebook側の補助関数。

| モジュール（ai_data_scientist配下） | 直接呼び出した関数・型 | 使用目的 |
|---|---|---|
| project_manager | resolve_project, ensure_notebook, ensure_data_dir, enqueue_write | 保存先の固定・原本配置・文書/メタデータの直列書込 |
| language_router | detect_language | 日本語判定 |
| lifecycle | register_run, mark_execution_start/end, mark_write_start/end, is_cancel_requested, mark_completed, mark_failed, get_run_status, wait_for_quiescence | 実行・書込フェーズと完了/失敗状態 |
| ingestion | SourceSpec, ingest | ローカルCSV読込・行数制限 |
| cleaning | clean_dataset | 完全重複除去と影響記録 |
| eda | explore | 型・欠測・統計要約 |
| data_definition | FieldValue, build_manifest, DataDefinitionManifest.unresolved_fields | 来歴・単位・定義の信頼度記録 |
| analysis_assumptions | Assumption, AnalysisAssumptionManifest, check_manifest | 分析前提・因果範囲と未解決事項検査 |
| data_quality | detect_anomalies | 範囲・半球・非欠測・一意キーの意味的検査 |
| stats_analysis | correlation | 年と年平均の記述的相関。独立標本p値は主結論に使わない |
| sensitivity | SensitivityPlan, run_sensitivity | 24仕様の結論安定性 |
| visualization | render_chart, build_image_output | 日本語図5点と画像MIME保存 |
| insight_engine | extract_cited_value, record_insight | 実行済みMIME出力からの正確な引用と根拠manifest |
| notebook_audit | audit_notebook, audit_visual_outputs | 実行・根拠・visual_audit=True、および改善候補の最小再現 |

mcp_gateway.run_and_recordは未使用。このホストは既に外側のJupyter MCPツールを提供し、Notebook内へexecute(code)クライアントを注入するインターフェースはない。架空のMCPClientやローカルexecで代用せず、実行・出力保存はネイティブjupyter MCPのinsert_execute_code_cell/execute_cellを使用した。文書・根拠・チャートメタデータ変更はenqueue_write経由。'''
    improvements_text='''## Jupytermind改善候補
外部サービス問題と区別して以下2候補を記録する。ソースコードの修正・Issue登録は本依頼の対象外。

| 分類 | 再現条件 | 期待する挙動 | 実際の挙動 | 影響 | 回避策 | 関係セル・ログ |
|---|---|---|---|---|---|---|
| 機能不足：意味定義の未確認状態の一覧 | FieldValue(status="inferred")だけを持つmanifestでunresolved_fields() | unknownだけでなくinferredも一括把握できる専用APIまたはオプション | []を返す。実装docstring上はunknownのみであり、実装違反ではなくSkillの留保要求とのインターフェース差 | 推定された単位・定義が確認済みと誤解され得る | unknown一覧に加えinferredを明示走査・出力。本分析で実施 | 「時間軸・単位・意味的異常検査」セル、最終セルinferred_not_in_unresolved_fields=True |
| 監査不足：チャートメタデータ欠落 | 有効PNGを持つコードセルでmetadata.chartを完全に省略しaudit_visual_outputs | タイトル・軸・凡例・glyph情報が確認不能と警告 | ()を返し、メタデータが空だとラベル検査に入らない。record_chart/build_image_outputも自動でchart情報は付けない | visual_audit=Trueが未確認チャートを合格させ得る | 実描画時の警告捕捉と各図の仕様を保存し、最終書込でchartメタデータを明示補完。全5画像を目視確認 | 「初回図表」「反復2」セル・最終セルmissing_chart_metadata_not_flagged=True、chart-manifest.json |

切り分け：Kaggle認証・検索・CSV取得は成功。NSIDCの定義解説URL取得のHTTPErrorは外部HTTPアクセスの問題で、Jupytermind不具合ではない（生の資格情報・ヘッダは保存しない）。Jupyter MCPのcreateは親ディレクトリ不在で失敗したため一時Notebookでensure_notebookを呼び解消した。filesystem MCPの許可ディレクトリ制限は別ツール設定であり、内容除外ポリシーやJupytermindの問題ではない。ベンチマークランナーは呼び出しておらず評価不能。'''
    limits_text='''## どこまで結論できるか・適用しなかった機能
このCSVの北半球extentでは、長期的減少・3月最大/9月最小の季節性・夏のより大きな減少を記述できる。原因別寄与、人為温暖化の効果量、将来の無氷年、最新2026年の状態は識別できない。元CSV末尾は2019-05-31であり、長期解析は1979–2018年の完全年38年（1987/1988除外）。観測年をまたぐ線形トレンドは期間要約であって毎年の単調変化や一定速度を保証しない。

Extentの百万km²単位はKaggleデータ辞書でverified。15%閾値の詳細と受動マイクロ波/NASA Teamの測定経路は値/パスからinferredで、公的解説のHTTP取得が失敗したためverifiedに昇格していない。Missingの詳細定義・ライセンスはunknown。北半球Missingは全行0だが、日付列に42日ギャップがあり「Missing=0だから欠測なし」ではない。定義の未確認事項は測定精度の解釈を制約するが、掲載Extentの単位付き時間的変化の記述とは区別する。

隔日観測をむやみに日次補間しない。月観測率閾値は45%/90%、移行月・ギャップ月は厳格に除外。感度では40%/60%も検査した。年平均は月等重みを主とし日数重みを比較。主分解は月固定効果+線形傾向の加法モデルで、STL/季節ARIMAは不連続系列の補間・欠測モデルを新たに仮定する必要があり本目的に追加価値が小さいため使わない。固定季節が厳密に一定という前提は棄却し、月別傾向と3月–9月差を最終結論に併記した。3月–9月差は日次の年間最大–最小ではない。

HACは自己相関を考慮した近似区間で、欠落年を詰めた観測年ラグと暦年ラグの差は残る。暦年5年区切りwild bootstrapは8ブロックだけであり、ブロック間独立性や厳密な被覆率は保証しない。観測誤差・センサー/製品版変更の独立検証は未実施。

validate_anomalies/compare_datasetsは独立参照データが提供/ロードされていないため未使用。別のKaggleコピーやseaborn版を独立観測とみなさない。Kaggleから取得できたため指定の公開CSVフォールバックは使わない。統計zスコアだけの異常検出は強い季節性を誤検知するため不採用、意味的制約と暦日キー/間隔を検査した。機械学習・因果推論・将来予測もこの記述目的には不適用。'''
    history_text='## 反復依頼履歴\n初回の根拠表・図を読み直し、結論に実質的影響を持つ点だけを最大3回追加した。追加依頼文は下記に原文のまま保存する。\n'
    for item in iteration_history:
        history_text += '\n### 反復 '+str(item['iteration'])+'\n\n'+item['request_original']+'\n\n選定理由：'+item['reason']+'\n\n結果・証拠：'+item['evidence']+'\n```json\n'+json.dumps(item['result'],ensure_ascii=False,indent=2)+'\n```\n残る限界：'+item['limitations']+'\n'
    history_text += '\n停止理由：3回の上限に達した。主要な方向と季節差は維持され、以降の重要な不確実性は追加データ/観測定義の検証を要する。単なる当てはめ追加では解決しない。'
    sections=[('modules',modules_text),('improvements',improvements_text),('limits',limits_text),('iterations',history_text)]
    with phase('Notebook文書・チャート情報書込',writing=True):
        def refresh(nb):
            nb.cells=[c for c in nb.cells if not c.metadata.get('exp22_generated')]
            for cell in nb.cells:
                if cell.cell_type=='code' and cell.source.startswith('# 初回図表:'):
                    specs=chart_metadata[:3]
                    cell.metadata['chart']={**specs[0],'title':' / '.join(s['title'] for s in specs),'legend':[s['legend'] for s in specs],'specifications':specs}
                if cell.cell_type=='code' and cell.source.startswith('# 反復2:'):
                    specs=chart_metadata[3:]
                    cell.metadata['chart']={**specs[0],'title':' / '.join(s['title'] for s in specs),'legend':[s['legend'] for s in specs],'specifications':specs}
            for label,text in sections:
                nb.cells.append(nbformat.v4.new_markdown_cell(text,metadata={'exp22_generated':label}))
            nb.metadata['v020_exp22']={'run_id':RUN_ID,'provenance':provenance,'chart_count':len(chart_metadata),'versions':versions}
        project_manager.enqueue_write(handle,refresh)
        for cited,text,kind in extracted_claims:
            insight_engine.record_insight(handle,text,evidence_execution_count,cited,kind,language='ja')
        def order_final(nb):
            for cell in nb.cells:
                if cell.cell_type=='markdown' and '```evidence' in cell.source:
                    cell.metadata['exp22_generated']='insight'
            final_cells=[c for c in nb.cells if c.cell_type=='code' and c.source.startswith('# 最終記録・図表監査')]
            nb.cells=[c for c in nb.cells if c not in final_cells]+final_cells
        project_manager.enqueue_write(handle,order_final)
        (data_dir/'chart-manifest.json').write_text(json.dumps(chart_metadata,ensure_ascii=False,indent=2))
with phase('notebook_audit visual_audit=True'):
    audit_report=notebook_audit.audit_notebook(handle.notebook_path,visual_audit=True)
    print('NOTEBOOK_AUDIT:',json.dumps({**asdict(audit_report),'ok':audit_report.ok},ensure_ascii=False,indent=2))
    if not audit_report.ok:
        raise RuntimeError('Notebook監査失敗。findingsを修正する必要があります。')
with phase('監査と実行ログ保存',writing=True):
    (data_dir/'notebook-audit.json').write_text(json.dumps({**asdict(audit_report),'ok':audit_report.ok},ensure_ascii=False,indent=2))
lifecycle.mark_completed(RUN_ID)
status=lifecycle.wait_for_quiescence(RUN_ID,timeout_s=5)
assert status.state=='completed' and status.active_cell_executions==0 and status.pending_notebook_writes==0 and status.locks_held==0
(data_dir/'lifecycle.json').write_text(json.dumps({'status':asdict(status),'phase_log':phase_log},ensure_ascii=False,indent=2))
print('LIFECYCLE_FINAL:',json.dumps(asdict(status),ensure_ascii=False))
"""
final_source = final_source.replace(r'\\n', r'\n').replace(r'\\s', r'\s')
header = '''# 北極海氷：長期傾向と季節変動の分離
run_id: `v020-exp-22` / AI Data Scientist Skill / 全分析・APIコードはJupyter MCP実行。

## 分析計画
1. Kaggle SDKで公開候補を検索し、提供主体と内容を確認して選定。日付・単位・半球・観測間隔・欠測を検査する。
2. 月平均を作り、全12か月が採用できる年の年平均で長期傾向を示す。月固定効果+線形傾向で平均的季節成分と残差を分離する。
3. 年平均傾向・季節曲線・季節調整後の時系列を図示し、根拠付きInsightを記録する。
4. 初回結果を読み直し、重み/期間/頑健推定、季節形の変化、自己相関の影響を最大3回追加検証する。
5. カーネル再起動後、全コードセルを上からJupyter MCPで再実行。原本SHA・初回数値との一致・図/Notebook監査・ライフサイクルを確認する。

## データ選定と来歴
Kaggle参照検索：https://www.kaggle.com/datasets?search=arctic+sea+ice+extent

選定：`nsidcorg/daily-sea-ice-extent-data` の `seaice.csv`。他の転載候補よりNSIDC掲載を優先。取得成功のためseaborn公開CSVフォールバックは未使用。取得日時・SHA-256は取得セルとdata/provenance.jsonに記録。元CSVを変更せず保存し、再実行ではSHAを検証した固定原本を再利用する。認証トークンは変数表示/ログ/Notebookに出さない。

Kaggleの説明文は1978–2015年だが、実CSVは1978-10-26〜2019-05-31年。北半球のみ13177観測、南半球を混合しない。単位はKaggle定義の10^6 km²（百万km²）。最終結論の適用期間は最新年ではなく1979–2018年である。
'''
def organize(nb):
    nb.cells=[c for c in nb.cells if not(c.cell_type=='code' and not c.source.strip())]
    nb.cells.insert(0,nbformat.v4.new_markdown_cell(header))
    for cell in nb.cells:
        if cell.cell_type=='code' and cell.source.startswith('# 初回図表:'):
            cell.source=cell.source.replace("show_chart(annual, 'line'", "annual_plot=annual.set_index('year').reindex(range(1979,2019)).rename_axis('year').reset_index()\n    show_chart(annual_plot, 'line'")
        if cell.cell_type=='code' and cell.source.startswith('# 反復2:'):
            cell.source=cell.source.replace("show_chart(amplitude,'line'", "amplitude_plot=amplitude.set_index('year').reindex(range(1979,2019)).rename_axis('year').reset_index()\n    show_chart(amplitude_plot,'line'")
    nb.cells.append(nbformat.v4.new_code_cell(final_source))
with phase('最終セルと計画節の書込',writing=True):
    project_manager.enqueue_write(handle,organize)
print('計画・全分析セル・最終記録セルを保存:',handle.notebook_path)
print('全コードセル再実行の前にカーネルを再起動する。')

計画・全分析セル・最終記録セルを保存: /home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-22-seaice/notebooks/kaggle-v020-kaggle-exp-22-seaice.ipynb
全コードセル再実行の前にカーネルを再起動する。
